# 센과 치히로의 행방불명 센과 치히로의 행방불명 ost를 활용한 조바꿈 탐구

"언제나 몇 번이라도" 라는 곡을 활용해 조바꿈이 진행되었을 때, (1) 12 평균율, (2) 피타고라스 음율, (3) 프톨레마이오스 음율 에 따라 음이 어떻게 들리는지를 확인해보는 code를 작성해봤습니다.

뻘짓이지만, '이렇게 해보면, 12 평균율의 우수성을 청각으로 직접 느낄 수 있지 않을까?' 라는 생각에서 시작된 code로, 앞으로 조금 더 개선될 여지가 보입니다.

이 code는 https://colab.research.google.com/drive/194D1GcoZl_d52_Tt5DdwQhj1hkpNMqtd?usp=sharing 에서 실행해보실 수 있습니다.

# 함수 선언

함수는 총 5가지 선언했습니다.

1. 주파수_12평균율: 12평균율에 맞춘 주파수 생성
2. 주파수_피타고라스: 피타고라스 음율에 맞춘 주파수 생성
3. 주파수_프톨레마이오스: 프톨레마이오스 음율에 맞춘 주파수 생성
4. generate_music_tone: 악보 (여기에서는 list)를 넣어주면 음을 생성해주는 함수
5. transpose: 조바꿈을 해주는 함수

잘 이해가 안된다면, ChatGPT에게 물어보시는게 빠를겁니다.

In [ ]:
import numpy as np
import IPython.display as ipd
import scipy.io.wavfile as wav





def 주파수_12평균율(계이름, A4_freq=440.0, 옥타브=1):
    """
    현대 12평균율을 사용하여 계이름에 해당하는 주파수를 계산합니다.

    :param 계이름: 음계 이름 (예: '도', '레#', '라')
    :param A4_freq: 기준 A4의 주파수 (Hz), 기본값은 440Hz
    :return: 계산된 주파수 (Hz)
    """
    note_map = {
        '도': -9, '도#': -8, '레b': -8, '레': -7, '레#': -6, '미b': -6,
        '미': -5, '파': -4, '파#': -3, '솔b': -3, '솔': -2, '솔#': -1, '라b': -1,
        '라': 0, '라#': 1, '시b': 1, '시': 2
    }

    if 계이름 not in note_map:
        raise ValueError("Invalid note name")

    n = note_map[계이름]
    return A4_freq * (2 ** (n / 12)) * 옥타브

def 주파수_피타고라스(계이름, A4_freq=440.0, 옥타브=1):
    """
    피타고라스 음률을 사용하여 주파수를 계산합니다.

    :param 계이름: 음계 이름 (예: '도', '레#', '라')
    :param A4_freq: 기준 A4의 주파수 (Hz)
    :return: 계산된 주파수 (Hz)
    """
    pythagorean_ratios = {
        '도': 1, '레': 9/8, '미': 81/64, '파': 4/3, '솔': 3/2, '라': 27/16, '시': 243/128,
        '도#': 256/243, '레#': 32/27, '미b': 32/27, '파#': 729/512, '솔#': 128/81, '라#': 16/9, '시b': 16/9
    }

    if 계이름 not in pythagorean_ratios:
        raise ValueError("Invalid note name")

    # C4_freq = A4_freq / (2 ** (9/12))  # C4 계산
    C4_freq = A4_freq / (27/16)  # C4 계산
    return C4_freq * pythagorean_ratios[계이름] * 옥타브

def 주파수_프톨레마이오스(계이름, A4_freq=440.0, 옥타브=1):
    """
    프톨레마이오스 음률을 사용하여 주파수를 계산합니다.

    :param 계이름: 음계 이름 (예: '도', '레#', '라')
    :param A4_freq: 기준 A4의 주파수 (Hz)
    :return: 계산된 주파수 (Hz)
    """
    ptolemaic_ratios = {
        '도': 1, '레': 9/8, '미': 5/4, '파': 4/3, '솔': 3/2, '라': 5/3, '시': 15/8,
        '도#': 16/15, '레#': 6/5, '미b': 6/5, '파#': 45/32, '솔#': 8/5, '라#': 9/5, '시b': 9/5
    }

    if 계이름 not in ptolemaic_ratios:
        raise ValueError("Invalid note name")

    # C4_freq = A4_freq / (2 ** (9/12))  # C4 계산
    C4_freq = A4_freq / (5/3)  # C4 계산
    return C4_freq * ptolemaic_ratios[계이름] * 옥타브

def generate_music_tone(note_list, scale_function, sample_rate=44100, bit_depth=32):
    import numpy as np
    import scipy.io.wavfile as wav
    import IPython.display as ipd

    total_duration = sum([dur for _, dur, *_ in note_list])
    t = np.linspace(0, total_duration, int(sample_rate * total_duration), endpoint=False)
    wave = np.zeros_like(t)

    current_pos = 0
    for note in note_list:
        계이름, duration, *옥타브 = note
        start_sample = int(current_pos * sample_rate)
        end_sample = int((current_pos + duration) * sample_rate)

        if 계이름 == "쉼":
            pass
        else:
            옥 = 옥타브[0] if 옥타브 else 1
            freq = scale_function(계이름, 옥타브=옥)
            wave[start_sample:end_sample] += 0.5 * np.sin(2 * np.pi * freq * t[start_sample:end_sample])

        current_pos += duration

    if bit_depth == 16:
        wave_int = (wave * 32767).astype(np.int16)
        filename = "snippet_16bit.wav"
    elif bit_depth == 24:
        wave_int = (wave * 8388607).astype(np.int32)
        filename = "snippet_24bit.wav"
    elif bit_depth == 32:
        wave_int = wave.astype(np.float32)
        filename = "snippet_32bit.wav"
    else:
        raise ValueError("지원되지 않는 비트 깊이")

    wav.write(filename, sample_rate, wave_int)
    return ipd.Audio(filename)

def transpose(note_list, 반음수):
    음계 = ['도', '도#', '레', '레#', '미', '파', '파#', '솔', '솔#', '라', '라#', '시']

    transposed_list = []

    for note in note_list:
        if note[0] == "쉼":
            transposed_list.append(note)
            continue

        계이름, 길이, 옥타브 = note

        # '미b', '시b' 같은 경우는 표준화 필요
        enharmonic_map = {
            '레b': '도#', '미b': '레#', '솔b': '파#', '라b': '솔#', '시b': '라#'
        }
        계이름 = enharmonic_map.get(계이름, 계이름)

        try:
            index = 음계.index(계이름)
        except ValueError:
            raise ValueError(f"계이름 '{계이름}' 인식 불가")

        new_index = index + 반음수
        octave_shift = new_index // 12
        new_index %= 12
        new_note = 음계[new_index]

        transposed_list.append((new_note, 길이, 옥타브 + octave_shift))

    return transposed_list

# 악보 만들기

악보는... 제 귀로 듣고 끄적거리면서 써봤습니다.

그래서 실제 곡과 조금 다를 수 있습니다.

In [ ]:
# Itumo Nando Demo (언제나 몇번이라도)
악보_1 = [
    ("파", 0.25, 1),
    ("솔", 0.25, 1),
    ("라", 0.25, 1),
    ("파", 0.25, 1),
    ("도", 0.75, 2),
    ("라", 0.25, 1),
    ("솔", 0.5, 1),
    ("도", 0.5, 2),
    ("솔", 0.5, 1),
    ("파", 0.25, 1),
    ("레", 0.25, 1),
    ("라", 0.75, 1),
    ("파", 0.25, 1),
    ("미", 0.5, 1),
    ("쉼", 0.5, 1),

    ("파", 0.25, 1),
    ("미", 0.25, 1),
    ("레", 0.25, 1),
    ("레", 0.05, 1),
    ("레", 0.2, 1),
    ("미", 0.5, 1),
    ("파", 0.25, 1),
    ("솔", 0.25, 1),
    ("도", 0.5, 1),
    ("파", 0.5, 1),
    ("솔", 0.25, 1),
    ("라", 0.25, 1),
    ("시b", 0.45, 1),
    ("쉼", 0.05),
    ("시b", 0.25, 1),
    ("라", 0.25, 1),
    ("솔", 0.25, 1),
    ("파", 0.25, 1),
    ("솔", 0.5, 1),
    ("쉼", 0.5, 1),

    ("파", 0.25, 1),
    ("솔", 0.25, 1),
    ("라", 0.25, 1),
    ("파", 0.25, 1),
    ("도", 0.75, 2),
    ("라", 0.25, 1),
    ("솔", 0.5, 1),
    ("도", 0.5, 2),
    ("솔", 0.5, 1),
    ("파", 0.25, 1),
    ("레", 0.25, 1),
    ("쉼", 0.05),
    ("레", 0.45, 1),
    ("미", 0.25, 1),
    ("파", 0.25, 1),
    ("도", 0.5, 1),
    ("쉼", 0.5, 1),

    ("도", 0.5, 1),
    ("레", 0.5, 1),
    ("미", 0.5, 1),
    ("파", 0.25, 1),
    ("솔", 0.25, 1),
    ("도", 0.5, 1),
    ("파", 0.5, 1),
    ("솔", 0.25, 1),
    ("라", 0.25, 1),
    ("시b", 0.45, 1),
    ("쉼", 0.05),
    ("시b", 0.25, 1),
    ("라", 0.25, 1),
    ("솔", 0.25, 1),
    ("파", 0.25, 1),
    ("쉼", 0.05),
    ("파", 0.70, 1),
    ("쉼", 1),
    ("라", 0.25, 1),
    ("시b", 0.25, 1),
    ("도", 0.45, 2),
    ("쉼", 0.05),
    ("도", 0.45, 2),
    ("쉼", 0.05),
    ("도", 0.45, 2),
    ("쉼", 0.05),
    ("도", 0.45, 2),
    ("쉼", 0.05),
    ("도", 0.25, 2),
    ("레", 0.25, 2),
    ("도", 0.25, 2),
    ("시b", 0.25, 1),

    ("라", 0.45, 1),
    ("쉼", 0.05),
    ("라", 0.45, 1),
    ("쉼", 0.05),
    ("라", 0.45, 1),
    ("쉼", 0.05),
    ("라", 0.45, 1),
    ("쉼", 0.05),
    ("라", 0.25, 1),
    ("시b", 0.25, 1),
    ("라", 0.25, 1),
    ("솔", 0.25, 1),
    ("파", 0.45, 1),
    ("쉼", 0.05),
    ("파", 0.40, 1),
    ("쉼", 0.05),
    ("파", 0.25, 1),
    ("미", 0.25, 1),
    ("레", 0.5, 1),
    ("미", 0.45, 1),
    ("쉼", 0.05),
    ("미", 0.25, 1),
    ("파", 0.25, 1),
    ("솔", 0.45, 1),
    ("쉼", 0.05),
    ("솔", 0.25, 1),
    ("라", 0.25, 1),
    ("솔", 0.25, 1),
    ("라", 0.25, 1),
    ("솔", 0.8, 1),
    ("쉼", 0.2),


    ("라", 0.25, 1),
    ("시b", 0.25, 1),
    ("도", 0.45, 2),
    ("쉼", 0.05),
    ("도", 0.45, 2),
    ("쉼", 0.05),
    ("도", 0.45, 2),
    ("쉼", 0.05),
    ("도", 0.45, 2),
    ("쉼", 0.05),
    ("도", 0.25, 2),
    ("레", 0.25, 2),
    ("도", 0.25, 2),
    ("시b", 0.25, 1),

    ("라", 0.45, 1),
    ("쉼", 0.05),
    ("라", 0.45, 1),
    ("쉼", 0.05),
    ("라", 0.45, 1),
    ("쉼", 0.05),
    ("라", 0.45, 1),

    ("쉼", 0.1),
    ("라", 0.25, 1),
    ("시b", 0.25, 1),
    ("라", 0.25, 1),
    ("솔", 0.25, 1),
    ("파", 0.25, 1),
    ("미", 0.25, 1),


    ("레", 0.45, 1),
    ("쉼", 0.05),
    ("레", 0.25, 1),
    ("미", 0.25, 1),
    ("파", 0.25, 1),
    ("솔", 0.25, 1),
    ("도", 0.5, 1),
    ("파", 0.5, 1),
    ("솔", 0.25, 1),
    ("라", 0.25, 1),
    ("솔", 0.70, 1),
    ("쉼", 0.05),
    ("솔", 0.20, 1),
    ("쉼", 0.05),
    ("솔", 0.25, 1),
    ("파", 0.20, 1),
    ("쉼", 0.05),
    ("파", 1.5, 1),

]

이제 위에서 선언한 함수를 활용해 센과 치히로의 행방불명 ost를 들어봅시다.

차이가 느껴지시나요?

#조바꿈이 없을 때 들리는 소리

In [ ]:
generate_music_tone(악보_1, 주파수_12평균율)

In [ ]:
generate_music_tone(악보_1, 주파수_피타고라스)

In [ ]:
generate_music_tone(악보_1, 주파수_프톨레마이오스)

# 조바꿈을 주면서 변화 1.

In [ ]:
generate_music_tone(transpose(악보_1,1), 주파수_12평균율)

In [ ]:
generate_music_tone(transpose(악보_1,1), 주파수_피타고라스)

In [ ]:
generate_music_tone(transpose(악보_1,1), 주파수_프톨레마이오스)

# 조바꿈을 주면서 변화 2.

In [ ]:
generate_music_tone(transpose(악보_1,2), 주파수_12평균율)

In [ ]:
generate_music_tone(transpose(악보_1,2), 주파수_피타고라스)

In [ ]:
generate_music_tone(transpose(악보_1,2), 주파수_프톨레마이오스)

# 조바꿈을 주면서 변화 3.

In [ ]:
generate_music_tone(transpose(악보_1,3), 주파수_12평균율)

In [ ]:
generate_music_tone(transpose(악보_1,3), 주파수_피타고라스)

In [ ]:
generate_music_tone(transpose(악보_1,3), 주파수_프톨레마이오스)

# 조바꿈을 주면서 변화 4.

In [ ]:
generate_music_tone(transpose(악보_1,4), 주파수_12평균율)

In [ ]:
generate_music_tone(transpose(악보_1,4), 주파수_피타고라스)

In [ ]:
generate_music_tone(transpose(악보_1,4), 주파수_프톨레마이오스)

# 조바꿈을 주면서 변화 5.

In [ ]:
generate_music_tone(transpose(악보_1,5), 주파수_12평균율)

In [ ]:
generate_music_tone(transpose(악보_1,5), 주파수_피타고라스)

In [ ]:
generate_music_tone(transpose(악보_1,5), 주파수_프톨레마이오스)

# 조바꿈을 주면서 변화 6.

In [ ]:
generate_music_tone(transpose(악보_1,6), 주파수_12평균율)

In [ ]:
generate_music_tone(transpose(악보_1,6), 주파수_피타고라스)

In [ ]:
generate_music_tone(transpose(악보_1,6), 주파수_프톨레마이오스)

# 조바꿈을 주면서 변화 7.

In [ ]:
generate_music_tone(transpose(악보_1,7), 주파수_12평균율)

In [ ]:
generate_music_tone(transpose(악보_1,7), 주파수_피타고라스)

In [ ]:
generate_music_tone(transpose(악보_1,7), 주파수_프톨레마이오스)